### Get Raw JSON Data
Bash Script

In [ ]:
#/bin/bash
%%bash
# Base API URLs
API_BASE_URL_MAKES="https://api.nhtsa.gov/products/vehicle/makes?modelYear="
API_BASE_URL_MODELS="https://api.nhtsa.gov/products/vehicle/models?"
API_BASE_URL_RECALLS="https://api.nhtsa.gov/recalls/recallsByVehicle?make=%s&model=%s&modelYear=%s"

# Get current year
CURRENT_YEAR=$(date +%Y)

# Calculate start year (last 5 years including current)
START_YEAR=$((CURRENT_YEAR - 9))

# Remove any existing JSON files to start fresh for this comprehensive run
rm -f year_*.json

echo "Starting comprehensive data fetch for model years from $START_YEAR to $CURRENT_YEAR..."

# Loop through the defined years
for (( YEAR=$START_YEAR; YEAR<=$CURRENT_YEAR; YEAR++ ))
do
    echo "\n--- Processing year: $YEAR ---"

    # 1. Fetch Makes for the current year
    API_URL_MAKE="${API_BASE_URL_MAKES}${YEAR}&issueType=r"
    RESPONSE_MAKE=$(curl -s "$API_URL_MAKE")

    MAKE_FILE="year_${YEAR}_makes.json"
    echo "$RESPONSE_MAKE" > "$MAKE_FILE"
    echo "Saved raw makes response for year $YEAR to $MAKE_FILE"

    # Check if the makes file exists and is not empty before processing
    if [ -s "$MAKE_FILE" ]; then
        # Extract unique makes using jq
        UNIQUE_MAKES=$(jq -r '.results[].make' "$MAKE_FILE" | sort -u)

        # Use IFS to correctly iterate over make names with spaces
        OLD_IFS=$IFS
        IFS=$'\n'
        for MAKE in $UNIQUE_MAKES
        do
            # URL-encode the make name
            ENCODED_MAKE_NAME=$(python3 -c 'import urllib.parse, sys; print(urllib.parse.quote_plus(sys.argv[1]))' "$MAKE")

            # 2. Fetch Models for the current year and make
            API_URL_MODEL="${API_BASE_URL_MODELS}modelYear=${YEAR}&make=${ENCODED_MAKE_NAME}&issueType=r"
            RESPONSE_MODEL=$(curl -s "$API_URL_MODEL")

            # Extract the 'Count' for models, default to 0 if null
            MODEL_COUNT=$(echo "$RESPONSE_MODEL" | jq '.count // 0')

            if [ "$MODEL_COUNT" -gt 0 ]; then
                SAFE_MAKE_NAME=$(echo "$MAKE" | tr ' ' '_')
                MODEL_FILE="year_${YEAR}_make_${SAFE_MAKE_NAME}_models.json"
                echo "$RESPONSE_MODEL" > "$MODEL_FILE"
                echo "  Saved models for year $YEAR, make '$MAKE' (Count: $MODEL_COUNT) to $MODEL_FILE"

                # Extract unique models from the current model file
                MODELS=$(jq -r '.results[].model' "$MODEL_FILE" | sort -u)

                if [ -z "$MODELS" ]; then
                    echo "  No distinct models found in $MODEL_FILE. Skipping recall fetching for this make.\n"
                    continue
                fi

                for MODEL in $MODELS
                do
                    # URL-encode the model name
                    ENCODED_MODEL_NAME=$(python3 -c 'import urllib.parse, sys; print(urllib.parse.quote_plus(sys.argv[1]))' "$MODEL")

                    # 3. Fetch Recalls for the current year, make, and model
                    API_URL_RECALL_SPECIFIC=$(printf "$API_BASE_URL_RECALLS" "$ENCODED_MAKE_NAME" "$ENCODED_MODEL_NAME" "$YEAR")
                    RESPONSE_RECALL=$(curl -s "$API_URL_RECALL_SPECIFIC")

                    # Extract the 'Count' for recalls, default to 0 if null
                    RECALL_COUNT=$(echo "$RESPONSE_RECALL" | jq '.Count // 0')

                    if [ "$RECALL_COUNT" -gt 0 ]; then
                        SAFE_MODEL_NAME=$(echo "$MODEL" | tr ' ' '_')
                        RECALL_FILE="year_${YEAR}_make_${SAFE_MAKE_NAME}_model_${SAFE_MODEL_NAME}_recalls.json"
                        echo "$RESPONSE_RECALL" > "$RECALL_FILE"
                        echo "    Saved recalls for year $YEAR, make '$MAKE', model '$MODEL' (Count: $RECALL_COUNT) to $RECALL_FILE"
                    else
                        echo "    No recalls found for year $YEAR, make '$MAKE', model '$MODEL' (Count: $RECALL_COUNT). Skipping file save."
                    fi
                done
            else
                echo "  No models found for year $YEAR, make '$MAKE' (Count: $MODEL_COUNT). Skipping model and recall fetch.\n"
            fi
        done
        IFS=$OLD_IFS # Restore original IFS
    else
        echo "Warning: $MAKE_FILE is empty or does not exist. Skipping model and recall fetching for this year.\n"
    fi
done

### Put Raw Data Into Folders

In [ ]:
#/bin/bash
%%bash
# Get current year and start year from previous
CURRENT_YEAR=$(date +%Y)
START_YEAR=$((CURRENT_YEAR - 9))

echo "Organizing files for model years from $START_YEAR to $CURRENT_YEAR..."

# Loop through the defined years to create directories and move files
for (( YEAR=$START_YEAR; YEAR<=$CURRENT_YEAR; YEAR++ ))
do
    echo "\n--- Organizing data for year: $YEAR ---"

    # Create year-specific directories if they don't exist
    mkdir -p "data/${YEAR}/makes"
    mkdir -p "data/${YEAR}/models"
    mkdir -p "data/${YEAR}/recalls"

    # Move makes files
    if [ -f "year_${YEAR}_makes.json" ]; then
        mv "year_${YEAR}_makes.json" "data/${YEAR}/makes/"
        echo "Moved year_${YEAR}_makes.json to data/${YEAR}/makes/"
    fi

    # Move models files
    # The models files are named year_YYYY_make_MAKE_models.json
    for model_file in year_${YEAR}_make_*_models.json; do
        if [ -f "$model_file" ]; then
            mv "$model_file" "data/${YEAR}/models/"
            echo "Moved $model_file to data/${YEAR}/models/"
        fi
    done

    # Move recalls files
    # The recalls files are named year_YYYY_make_MAKE_model_MODEL_recalls.json
    for recall_file in year_${YEAR}_make_*_model_*_recalls.json; do
        if [ -f "$recall_file" ]; then
            mv "$recall_file" "data/${YEAR}/recalls/"
            echo "Moved $recall_file to data/${YEAR}/recalls/"
        fi
    done
done

Organizing files for model years from 2016 to 2025...
\n--- Organizing data for year: 2016 ---
Moved year_2016_makes.json to data/2016/makes/
Moved year_2016_make_ACURA_models.json to data/2016/models/
Moved year_2016_make_AIRSTREAM_models.json to data/2016/models/
Moved year_2016_make_AMERICAN_COACH_models.json to data/2016/models/
Moved year_2016_make_APRILIA_models.json to data/2016/models/
Moved year_2016_make_ASTON_MARTIN_models.json to data/2016/models/
Moved year_2016_make_AUDI_models.json to data/2016/models/
Moved year_2016_make_AVION_models.json to data/2016/models/
Moved year_2016_make_BIG_TEX_models.json to data/2016/models/
Moved year_2016_make_BISON_models.json to data/2016/models/
Moved year_2016_make_BLUE_BIRD_models.json to data/2016/models/
Moved year_2016_make_BMW_models.json to data/2016/models/
Moved year_2016_make_BRAUNABILITY_models.json to data/2016/models/
Moved year_2016_make_BUICK_models.json to data/2016/models/
Moved year_2016_make_CADILLAC_models.json to d

### Clean the Data
Put all the data into a single JSON dump named "all_recalls.json"

In [ ]:
# Don't do this if you ran all the bash scripts
%%bash
unzip dataRaw.zip

Archive:  dataRaw.zip
   creating: data/
   creating: data/2017/
   creating: data/2017/recalls/
  inflating: data/2017/recalls/year_2017_make_BMW_model_X4_recalls.json  
  inflating: data/2017/recalls/year_2017_make_TRIUMPH_model_STREET_TWIN_recalls.json  
  inflating: data/2017/recalls/year_2017_make_ACURA_model_NSX_recalls.json  
  inflating: data/2017/recalls/year_2017_make_ACURA_model_TLX_recalls.json  
  inflating: data/2017/recalls/year_2017_make_RAM_model_2500_recalls.json  
  inflating: data/2017/recalls/year_2017_make_LIVIN_LITE_model_QUICKSILVER_6.0_recalls.json  
  inflating: data/2017/recalls/year_2017_make_KENTUCKY_TRAILER_model_MV_recalls.json  
  inflating: data/2017/recalls/year_2017_make_FREIGHTLINER_model_M2_recalls.json  
  inflating: data/2017/recalls/year_2017_make_LINCOLN_model_NAVIGATOR_recalls.json  
  inflating: data/2017/recalls/year_2017_make_CHEVROLET_model_TRAVERSE_recalls.json  
  inflating: data/2017/recalls/year_2017_make_AUDI_model_Q7_recalls.json  
  

In [ ]:
import json
import re
import glob
# glob is used to find pathnames matching a specific pattern
recall_files = glob.glob("data/**/recalls/*.json", recursive=True)

# List to store all extracted recall information
all_recalls_data = []
for filenames in recall_files:
  with open(str(filenames)) as f:
    data = json.load(f)
    result = data['results']
    resultSet = []
    #print(json.dumps(result, indent=2))
    for indent in result:
      # Extract only the part of the component string before the first ':', ',', ' ' or ';'
      component_full = indent['Component']
      match = re.search(r'([^:,;]+)', component_full)
      comp = match.group(1) if match else component_full

      mYear = indent['ModelYear']
      make = indent['Make']
      model = indent['Model']
      resultSet.append({
          'Component': comp,
          'ModelYear': mYear,
          'Make': make,
          'Model': model
      })
    # Changed 'all' to 'all_recalls_data.extend(resultSet)' to actually store the results
    all_recalls_data.extend(resultSet)
    #print(json.dumps(resultSet, indent=2))

output_file_name = 'all_recalls.json'

with open(output_file_name, 'w') as f:
    json.dump(all_recalls_data, f, indent=4)

print(f"All recalls data saved to {output_file_name}")

All recalls data saved to all_recalls.json


## Load JSON Data
Save the file as a variable



In [1]:
import json

output_file_name = 'all_recalls.json'

with open(output_file_name, 'r') as f:
    recall_data = json.load(f)

print(f"Successfully loaded {len(recall_data)} recall entries from {output_file_name}.")

Successfully loaded 14761 recall entries from all_recalls.json.


###Initialize SQLite Connection



In [2]:
import sqlite3

db_name = 'recalls.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()
print(f"Connected to SQLite database: {db_name}")

Connected to SQLite database: recalls.db


###Initialize Recalls and Makes Tables

In [3]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS makes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS recalls (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    year TEXT NOT NULL,
    make_id INTEGER NOT NULL,
    model TEXT NOT NULL,
    component TEXT NOT NULL,
    FOREIGN KEY (make_id) REFERENCES makes(id)
);
''')

conn.commit()

##Fill The Tables

Parses through the all_recalls JSON file

Every unique make (manufacturer) gets assigned an Id and put into the makes table

The rest of the json data is put into the recalls table



In [4]:
for entry in recall_data:
    make_name = entry['Make']
    model_year = entry['ModelYear']
    model_name = entry['Model']
    component = entry['Component']

    # Insert unique make into the 'makes' table
    cursor.execute('INSERT OR IGNORE INTO makes (name) VALUES (?)', (make_name,))
    conn.commit()

    # Get the make_id for the inserted or existing make
    cursor.execute('SELECT id FROM makes WHERE name = ?', (make_name,))
    make_id = cursor.fetchone()[0]

    # Insert recall data into the 'recalls' table
    cursor.execute(
        'INSERT INTO recalls (year, make_id, model, component) VALUES (?, ?, ?, ?)',
        (model_year, make_id, model_name, component)
    )

conn.commit()

prints the the first 5 raw rows of each table

In [ ]:

db_name = 'recalls.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

print("\n--- First 5 rows from 'makes' table ---")
cursor.execute('SELECT * FROM makes LIMIT 5')
makes_rows = cursor.fetchall()
for row in makes_rows:
    print(row)

print("\n--- First 5 rows from 'recalls' table ---")
cursor.execute('SELECT * FROM recalls LIMIT 5')
recalls_rows = cursor.fetchall()
for row in recalls_rows:
    print(row)

conn.close()


--- First 5 rows from 'makes' table ---
(1, 'AUDI')
(4, 'FORD')
(7, 'VOLKSWAGEN')
(16, 'FOREST RIVER')
(23, 'PORSCHE')

--- First 5 rows from 'recalls' table ---
(1, '2016', 1, 'A4', 'ENGINE AND ENGINE COOLING')
(2, '2016', 1, 'A4', 'ELECTRICAL SYSTEM')
(3, '2016', 1, 'A4', 'ENGINE AND ENGINE COOLING')
(4, '2016', 4, 'E-350', 'SEATS')
(5, '2016', 4, 'E-350', 'ELECTRICAL SYSTEM')


###Basic Pandas Dataframes

In [5]:
import pandas as pd
con = sqlite3.connect("recalls.db")
query_recalls = "SELECT * FROM recalls;"
query_makes = "SELECT * FROM makes"
recalls_df = pd.read_sql_query(query_recalls, con)
makes_df = pd.read_sql_query(query_makes, con)

In [6]:
recalls_df.to_csv('recalls_df.csv', index=False)
print("recalls_df saved to recalls_df.csv")

recalls_df saved to recalls_df.csv


In [7]:
makes_df.to_csv('makes_df.csv', index=False)
print("makes_df saved to makes_df.csv")

makes_df saved to makes_df.csv


In [ ]:
makes_df.head()

,id,name
0,1,AUDI
1,4,FORD
2,7,VOLKSWAGEN
3,16,FOREST RIVER
4,23,PORSCHE


###Most Common Manufacturers

This query gets the most common manufacturers that have recalls for their models in the last 10 years

In [ ]:
recalls_var = 'recalls.db'
conn = sqlite3.connect(recalls_var)

# SQL query to count components for each make_id
query_component_counts = """
SELECT
    m.name AS make_name,
    COUNT(r.component) AS component_count
FROM
    makes AS m
JOIN
    recalls AS r ON m.id = r.make_id
GROUP BY
    m.name
ORDER BY
    component_count DESC
"""

# Load the results into a pandas DataFrame
component_counts_df = pd.read_sql_query(query_component_counts, conn)

# Display the first 5 rows of the new DataFrame
display(component_counts_df.head(10))

conn.close()

,make_name,component_count
0,FORD,1110
1,FOREST RIVER,662
2,TOYOTA,568
3,BMW,560
4,CHEVROLET,473
5,HONDA,443
6,RAM,433
7,AUDI,430
8,NEWMAR,408
9,VOLKSWAGEN,371


### Major Trends
Question: What are the major trends in vehicle safety recalls in the United States?
<br><br>
Purpose: To show the most common recall type of all vehicals in the United States

In [ ]:
db_name = 'recalls.db'
conn = sqlite3.connect(db_name)

# SQL query to count all components and order by count DESC
query_all_component_counts = """
SELECT
    r.component,
    COUNT(r.component) AS component_count
FROM
    recalls AS r
GROUP BY
    r.component
ORDER BY
    component_count DESC
"""

# Load the results into a pandas DataFrame
all_component_counts_df = pd.read_sql_query(query_all_component_counts, conn)

display(all_component_counts_df.head(10))

conn.close()

,component,component_count
0,EQUIPMENT,2219
1,ELECTRICAL SYSTEM,1921
2,SERVICE BRAKES,1078
3,FUEL SYSTEM,1066
4,AIR BAGS,905
5,POWER TRAIN,874
6,BACK OVER PREVENTION,808
7,STRUCTURE,715
8,EXTERIOR LIGHTING,688
9,STEERING,640


###FORD Manufacure Results

This query takes the most common manufacturer (FORD) and makes a component count data frame.

This counts FORD vehical components from the last 10 years that get recalled and shows the most common component that is recalled.

In [ ]:
db_name = 'recalls.db'
conn = sqlite3.connect(db_name)

# SQL query to count components for 'FORD' and order by count DESC
query_ford_component_counts = """
SELECT
    r.component,
    COUNT(r.component) AS component_count
FROM
    makes AS m
JOIN
    recalls AS r ON m.id = r.make_id
WHERE
    m.name = 'FORD'
GROUP BY
    r.component
ORDER BY
    component_count DESC
"""

# Load the results into a pandas DataFrame
ford_component_counts_df = pd.read_sql_query(query_ford_component_counts, conn)

display(ford_component_counts_df)

conn.close()

,component,component_count
0,BACK OVER PREVENTION,183
1,POWER TRAIN,147
2,FUEL SYSTEM,100
3,EQUIPMENT,97
4,ELECTRICAL SYSTEM,79
5,SEATS,65
6,VISIBILITY,47
7,EXTERIOR LIGHTING,44
8,STEERING,41
9,SERVICE BRAKES,41


from matplotlib import pyplot as plt
ford_component_counts_df['component_count'].plot(kind='hist', bins=20, title='component_count')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
ford_component_counts_df['component_count'].plot(kind='line', figsize=(8, 4), title='component_count')
plt.gca().spines[['top', 'right']].set_visible(False)

###CHEVROLET Manufacture Recalls

In [ ]:
db_name = 'recalls.db'
conn = sqlite3.connect(db_name)

# SQL query to count components for 'FORD' and order by count DESC
query_ford_component_counts = """
SELECT
    r.component,
    COUNT(r.component) AS component_count
FROM
    makes AS m
JOIN
    recalls AS r ON m.id = r.make_id
WHERE
    m.name = 'CHEVROLET'
GROUP BY
    r.component
ORDER BY
    component_count DESC
"""

# Load the results into a pandas DataFrame
ford_component_counts_df = pd.read_sql_query(query_ford_component_counts, conn)

display(ford_component_counts_df)

conn.close()

,component,component_count
0,AIR BAGS,85
1,SERVICE BRAKES,60
2,SEAT BELTS,56
3,POWER TRAIN,41
4,FUEL SYSTEM,39
5,ELECTRICAL SYSTEM,33
6,LATCHES/LOCKS/LINKAGES,21
7,TIRES,20
8,STRUCTURE,18
9,SEATS,14
